<a href="https://colab.research.google.com/github/yogeshsahu04/gemini-gen-ai-poc/blob/main/Gemini_api_candidate_count_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import json

# Load the response schema from the generated file
with open('response_schema.json', 'r') as f:
    response_schema_json_content = f.read()

prompt_content = f"""# Role Definition
role: Communication Surveillance Analyst
objective: Thoroughly evaluate if a given communication (chat, email, or message) triggers any surveillance rules related to banking/financial compliance.

# Input
- Text: contains message text body

# Output Schema (YAML Format)
# The output must be a YAML string structured according to the following JSON schema:
{response_schema_json_content}

# Notes
- If **no rules are detected**, only populate `MsgSumm` and `modelThoughts`.
- Accuracy is the main focus: rules should only be flagged if clearly present.
- Candidate Count: To request multiple distinct responses, add → "candidate_count: 3" (where N is the number of variations required).
- **Output Format:** The output must be raw YAML string, do not wrap it in markdown code fences.
- The prompt, rules, and message inputs are provided in YAML format.
- The message input may contain HTML tags, which should be ignored for content analysis.
- Ensure all YAML quoted strings are properly closed and lists are correctly formatted.
- **Token Count:** Do NOT include token details within the `modelThoughts` section of each candidate's output. Token details will be reported once at the top level of the final output.
"""

with open('systemPrompt.yaml', 'w') as f:
    f.write(prompt_content)

print('Created systemPrompt.yaml')

Created systemPrompt.yaml


In [38]:
evaluation_rules_content = """
evaluation rules:
- rule_id: ER120
  rule_name: Market Manipulation
  description: Detects conversations suggesting attempts to distort, misrepresent, or artificially influence market prices, client trades, or financial instruments.

- rule_id: ER121
  rule_name: Abusive Language
  description: Flags use of offensive, discriminatory, or threatening language directed at clients, colleagues, or institutions.

- rule_id: ER122
  rule_name: Misrepresentation of Numbers
  description: Identifies attempts to falsify, exaggerate, or conceal financial figures, trade volumes, or performance metrics.

- rule_id: ER123
  rule_name: Insider Information Disclosure
  description: Detects sharing of non-public, material information about securities, clients, or corporate actions that could lead to unfair trading advantage.

- rule_id: ER124
  rule_name: Client Manipulation
  description: Flags communications that pressure, mislead, or coerce clients into trades or financial decisions against their best interest.

- rule_id: ER125
  rule_name: Collusion
  description: Identifies discussions suggesting coordination with other parties to fix prices, rig bids, or manipulate markets.

- rule_id: ER126
  rule_name: Unauthorized Trade Discussion
  description: Detects conversations about executing trades or transactions outside approved channels, policies, or without proper authorization.

- rule_id: ER127
  rule_name: Fraudulent Intent
  description: Flags language indicating intent to deceive, commit fraud, or conceal material facts in financial dealings.

- rule_id: ER128
  rule_name: Regulatory Evasion
  description: Identifies attempts to bypass, ignore, or conceal activities from regulators, auditors, or compliance monitoring.

- rule_id: ER129
  rule_name: Conflict of Interest
  description: Detects communications suggesting personal gain at the expense of client interests or institutional integrity."""

with open('evalRules.yaml', 'w') as f:
    f.write(evaluation_rules_content)

print('Created evalRules.yaml')


Created evalRules.yaml


In [39]:
message_text_content = "text: { <html><b>Internal Memo:</b> Our Q4 earnings will be much higher than expected, but don't tell the public until next month. Buy more stock now!</html>\n<html><i>Client Call:</i> You really need to invest heavily in Company X; their stock is about to skyrocket based on some confidential info I just got. Trust me on this one.</html>}"

with open('msg-01.yaml', 'w') as f:
    f.write(message_text_content)

print('Created msg-01.yaml')


Created msg-01.yaml


In [43]:
response_schema_content = '''{
  "MsgSumm": "Short 1–2 line summary of the message text",
  "ruleDetections": [
    {
      "ruleId": "string",              // ID of evaluation rule (e.g., ER120)
      "ruleName": "string",            // Name of evaluation rule (e.g., Market Manipulation)
      "explanation": "string",         // Brief reason why the rule was detected
      "cite": [                        // List of specific lines/sentences where rule was detected
        "string",
        "string"
      ],
      "ruleDetected": true             // Boolean flag, always true if rule triggered
    }
  ],
  "modelThoughts": {
    "summary": "Brief description of reasoning process"
  }
}'''

with open('response_schema.json', 'w') as f:
    f.write(response_schema_content)

print('Created response_schema.json')

Created response_schema.json


In [41]:
model_config_content = """model_config:
  model_name: 'gemini-2.5-flash'
  generation_config:
    candidate_count: 3
    max_output_tokens: 4096
    temperature: 0.8
    top_p: 1.0
  safety_settings:
    - category: 'HARM_CATEGORY_HARASSMENT'
      threshold: 'BLOCK_NONE'
    - category: 'HARM_CATEGORY_HATE_SPEECH'
      threshold: 'BLOCK_NONE'
    - category: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'
      threshold: 'BLOCK_NONE'
    - category: 'HARM_CATEGORY_DANGEROUS_CONTENT'
      threshold: 'BLOCK_NONE'
"""

with open('modelConfig.yaml', 'w') as f:
    f.write(model_config_content)

print('Created modelConfig.yaml')

Created modelConfig.yaml


In [44]:
import google.generativeai as genai
import os
from google.colab import userdata
import json
import yaml # Import PyYAML

# Configure the Gemini API key
try:
    API_KEY = userdata.get('GEMINI_API_KEY1')
except AttributeError:
    API_KEY = os.environ.get('GEMINI_API_KEY1')

if not API_KEY:
    raise ValueError("GEMINI_API_KEY1 not found. Please set it in Colab secrets or environment variables.")

genai.configure(api_key=API_KEY)

def initialize_model(model_config_path='modelConfig.yaml'):
    """
    Initializes the Generative Model based on configuration from a YAML file.
    """
    with open(model_config_path, 'r') as f:
        model_config_data = yaml.safe_load(f)

    model_name = model_config_data['model_config']['model_name']
    generation_config = model_config_data['model_config']['generation_config']
    safety_settings = model_config_data['model_config']['safety_settings']

    model = genai.GenerativeModel(
        model_name,
        generation_config=generation_config,
        safety_settings=safety_settings
    )
    return model

def load_prompt_components(system_prompt_path='systemPrompt.yaml', eval_rules_path='evalRules.yaml', message_path='msg-01.yaml'):
    """
    Loads prompt components from respective YAML files.
    """
    with open(system_prompt_path, 'r') as f:
        prompt_input = f.read()

    with open(eval_rules_path, 'r') as f:
        evaluation_rule_input = f.read()

    with open(message_path, 'r') as f:
        message_text_input = f.read()

    return prompt_input, evaluation_rule_input, message_text_input

def generate_content_with_model(model, full_prompt):
    """
    Generates content using the Gemini model and counts prompt tokens.
    """
    prompt_token_count = 0
    try:
        prompt_token_count = model.count_tokens(full_prompt).total_tokens
    except Exception as e:
        print(f"Could not count prompt tokens: {e}")

    response = model.generate_content(full_prompt)
    return response, prompt_token_count

def process_model_candidates(model, response, prompt_token_count, message_base_id):
    """
    Processes each candidate from the model's response, parses YAML,
    and collects summary details.
    """
    all_output_details = []
    all_candidates_parsed_data = []
    total_response_tokens = 0

    if response.candidates:
        for i, candidate in enumerate(response.candidates):
            candidate_summary = {
                "candidate_index": i,
                "msgSummary": None,
                "rulesDetectedCount": 0, # Keeping count for a quick overview
                "ruleDetections": [],
                "modelThoughts": {
                    "summary": "Could not parse model thoughts from candidate text."
                }
            }

            try:
                generated_text = candidate.content.parts[0].text
                if generated_text.startswith('```yaml') and generated_text.endswith('```'):
                    generated_text = generated_text[len('```yaml'):-len('```')].strip()
                elif generated_text.startswith('```') and generated_text.endswith('```'):
                    generated_text = generated_text[len('```'):-len('```')].strip()

                parsed_candidate_text = yaml.safe_load(generated_text)
                all_candidates_parsed_data.append(parsed_candidate_text)

                candidate_summary["msgSummary"] = parsed_candidate_text.get("MsgSumm")
                candidate_summary["ruleDetections"] = parsed_candidate_text.get("ruleDetections", [])
                candidate_summary["rulesDetectedCount"] = len(candidate_summary["ruleDetections"])

                model_thoughts_from_parsed_text = parsed_candidate_text.get("modelThoughts", {})
                candidate_summary["modelThoughts"]["summary"] = model_thoughts_from_parsed_text.get("summary", "N/A")

                # Count tokens for this specific candidate's response
                candidate_token_count = model.count_tokens(generated_text).total_tokens
                total_response_tokens += candidate_token_count

            except yaml.YAMLError as e:
                error_msg = f"Error parsing YAML from model output {i+1}: {e}. Raw text: {generated_text}"
                print(error_msg)
                candidate_summary["msgSummary"] = error_msg
            except Exception as e:
                error_msg = f"An unexpected error occurred while processing candidate {i+1}: {e}"
                print(error_msg)
                candidate_summary["msgSummary"] = error_msg

            all_output_details.append(candidate_summary)
    else:
        # Handle case where no candidates are generated
        print("No candidates generated by the model.")

    return all_output_details, all_candidates_parsed_data, total_response_tokens

def save_and_summarize_results(all_candidates_parsed_data, all_output_details, message_base_id, prompt_token_count, total_response_tokens, response):
    """
    Saves combined YAML output and prints the final JSON summary.
    """
    output_data = {
        "tokenDetails": {
            "promptTokens": prompt_token_count,
            "responseTokens": total_response_tokens
        },
        "candidates": [], # This will remain empty as per current output_data structure
        "output_file": "",
        "candidates_summary": all_output_details
    }

    if all_candidates_parsed_data:
        combined_output_file_name = f"{message_base_id}-all-outputs.yaml"
        with open(combined_output_file_name, 'w') as f:
            yaml.dump(all_candidates_parsed_data, f, indent=2)
        print(f"Saved all candidate outputs to {combined_output_file_name}")
        output_data["output_file"] = combined_output_file_name
    elif response.prompt_feedback and response.prompt_feedback.block_reason:
        output_data["promptFeedback"] = {
            "blockReason": response.prompt_feedback.block_reason.name,
            "blockReasonCategory": response.prompt_feedback.block_reason_category.name if response.prompt_feedback.block_reason_category else None
        }
    else:
        output_data["message"] = "No candidates generated and no specific feedback."

    print(json.dumps(output_data, indent=2))

def main():
    """
    Main function to orchestrate the content generation and processing.
    """
    message_base_id = "msg-1"

    try:
        model = initialize_model()
        prompt_input, evaluation_rule_input, message_text_input = load_prompt_components()

        full_prompt = f"{prompt_input}\n{evaluation_rule_input}\n{message_text_input}"

        response, prompt_token_count = generate_content_with_model(model, full_prompt)

        all_output_details, all_candidates_parsed_data, total_response_tokens = \
            process_model_candidates(model, response, prompt_token_count, message_base_id)

        save_and_summarize_results(
            all_candidates_parsed_data, all_output_details, message_base_id,
            prompt_token_count, total_response_tokens, response
        )

    except Exception as e:
        print(f"An error occurred during content generation: {e}")
        if "API key not valid" in str(e) or "Authentication failed" in str(e):
            print("Please ensure your GEMINI_API_KEY is correctly set in Colab secrets and is valid.")
        else:
            print("Please check your network connection or try again later.")

if __name__ == '__main__':
    main()

Saved all candidate outputs to msg-1-all-outputs.yaml
{
  "tokenDetails": {
    "promptTokens": 950,
    "responseTokens": 2234
  },
  "candidates": [],
  "output_file": "msg-1-all-outputs.yaml",
  "candidates_summary": [
    {
      "candidate_index": 0,
      "msgSummary": "The message contains an internal memo advising to buy stock based on withheld positive Q4 earnings, and a client call recommending heavy investment in a company due to confidential information.",
      "rulesDetectedCount": 6,
      "ruleDetections": [
        {
          "ruleId": "ER120",
          "ruleName": "Market Manipulation",
          "explanation": "The communication suggests artificially influencing market prices by withholding positive earnings news while advising to buy stock.",
          "cite": [
            "Our Q4 earnings will be much higher than expected, but don't tell the public until next month. Buy more stock now!"
          ],
          "ruleDetected": true
        },
        {
          "